## Классификация: превышает ли значение SI значение 8

In [1]:
import pandas as pd

import sys
sys.path.append('../src')
from preprocessing import DatasetPreprocessor
from metrics_calculator import MetricsCalculator

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [3]:
df = pd.read_excel('../data/raw/classicMLCourseWorkData.xlsx')

# удаляем строки с пропусками
clean_df = df.dropna().copy()

# медиана SI
si_median = clean_df['SI'].median()
# бинарный таргет
y = (clean_df['SI'] > 8).astype(int)
# признаки
X = clean_df.drop(columns=['SI', 'IC50, mM', 'CC50, mM'])

# Класс расчета метрик
metrics_helper = MetricsCalculator()

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#### Логистическая регрессия

In [5]:
logreg_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column=None,
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

logreg_pipeline.fit(X_train, y_train)

logreg_y_train_pred = logreg_pipeline.predict(X_train)
logreg_y_test_pred = logreg_pipeline.predict(X_test)

logreg_train_metrics = metrics_helper.classification_metrics(y_train, logreg_y_train_pred)
logreg_test_metrics = metrics_helper.classification_metrics(y_test, logreg_y_test_pred)

print('Метрики на train:')
display(logreg_train_metrics)

print('Метрики на test:')
display(logreg_test_metrics)

Метрики на train:


{'accuracy': 0.7807017543859649,
 'precision': 0.7522935779816514,
 'recall': 0.5754385964912281,
 'f1': 0.6520874751491054}

Метрики на test:


{'accuracy': 0.69,
 'precision': 0.5616438356164384,
 'recall': 0.5774647887323944,
 'f1': 0.5694444444444444}

- Модель показывает среднее качество на train (accuracy $\approx 0.78$) и хуже на test (accuracy $\approx 0.69$)
- Метрики precision, recall и f1 на test находятся на низком уровне

**Вывод:** логистическая регрессия даёт слабое качество

#### Случайный лес

In [7]:
rf_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column=None,
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
])

# обучение
rf_pipeline.fit(X_train, y_train)

# предсказания
rf_y_train_pred = rf_pipeline.predict(X_train)
rf_y_test_pred = rf_pipeline.predict(X_test)

# метрики
rf_train_metrics = metrics_helper.classification_metrics(y_train, rf_y_train_pred)
rf_test_metrics = metrics_helper.classification_metrics(y_test, rf_y_test_pred)

print('Метрики на train:')
display(rf_train_metrics)

print('Метрики на test:')
display(rf_test_metrics)

Метрики на train:


{'accuracy': 0.943609022556391,
 'precision': 0.9347826086956522,
 'recall': 0.9052631578947369,
 'f1': 0.9197860962566845}

Метрики на test:


{'accuracy': 0.7,
 'precision': 0.5901639344262295,
 'recall': 0.5070422535211268,
 'f1': 0.5454545454545454}

- Модель показывает высокое качество на train (accuracy $\approx 0.94$), но хуже на test (accuracy $\approx 0.70$)
- Наблюдается переобучение
- Метрики на test остаются на низком уровне

**Вывод:** RandomForest не даёт существенного улучшения качества

In [8]:
# сетка гиперпараметров
rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 0.5]
}

# GridSearch
rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# обучение
rf_grid_search.fit(X_train, y_train)

# лучшая модель
best_rf_pipeline = rf_grid_search.best_estimator_

print('Лучшие параметры:')
print(rf_grid_search.best_params_)

# предсказания
best_rf_y_train_pred = best_rf_pipeline.predict(X_train)
best_rf_y_test_pred = best_rf_pipeline.predict(X_test)

# метрики
best_rf_train_metrics = metrics_helper.classification_metrics(y_train, best_rf_y_train_pred)
best_rf_test_metrics = metrics_helper.classification_metrics(y_test, best_rf_y_test_pred)

print('Метрики на train:')
display(best_rf_train_metrics)

print('Метрики на test:')
display(best_rf_test_metrics)

Лучшие параметры:
{'model__max_depth': None, 'model__max_features': 0.5, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Метрики на train:


{'accuracy': 0.9411027568922306,
 'precision': 0.9407407407407408,
 'recall': 0.8912280701754386,
 'f1': 0.9153153153153153}

Метрики на test:


{'accuracy': 0.695,
 'precision': 0.5833333333333334,
 'recall': 0.49295774647887325,
 'f1': 0.5343511450381679}

- Подбор гиперпараметров не улучшил качество на test (accuracy $\approx 0.70$, f1 $\approx 0.53$)
- Разрыв между train и test сохраняется

**Вывод:** настройка RandomForest не даёт улучшения

#### Градиентный бустинг

In [9]:
from sklearn.ensemble import GradientBoostingClassifier

# pipeline
gb_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column=None,
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        )
    )
])

# обучение
gb_pipeline.fit(X_train, y_train)

# предсказания
gb_y_train_pred = gb_pipeline.predict(X_train)
gb_y_test_pred = gb_pipeline.predict(X_test)

# метрики
gb_train_metrics = metrics_helper.classification_metrics(y_train, gb_y_train_pred)
gb_test_metrics = metrics_helper.classification_metrics(y_test, gb_y_test_pred)

print('Метрики на train:')
display(gb_train_metrics)

print('Метрики на test:')
display(gb_test_metrics)

Метрики на train:


{'accuracy': 0.8922305764411027,
 'precision': 0.8871595330739299,
 'recall': 0.8,
 'f1': 0.8413284132841329}

Метрики на test:


{'accuracy': 0.7,
 'precision': 0.582089552238806,
 'recall': 0.5492957746478874,
 'f1': 0.5652173913043478}

- Модель показывает высокое качество на train (accuracy $\approx 0.89$), но ниже на test (accuracy $\approx 0.70$)
- Наблюдается переобучение
- Качество сопоставимо с RandomForest

**Вывод:** GradientBoosting не даёт улучшения

- Все модели показывают низкое качество на test (accuracy $\approx 0.69$–$0.70$, f1 $\approx 0.53$–$0.57$)
- Улучшения от более сложных моделей отсутствуют

**Вывод:** задача SI > 8 решается с низким качеством,
результаты модели ненадёжны